# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir, type):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.{type}_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Chuẩn bị dữ liệu

In [35]:
df_user = read_parquet_user('./preprocessed-dataset')
df_user.head()

customer_id,gender,province,membership,created_date,install_app,install_datetime,user_age_days,days_since_install
i32,str,str,str,date,str,date,f64,f64
7892164,"""male""","""Đồng Nai""","""Standard""",2024-09-22,"""In-Store""",2024-09-22,373.425774,373.917091
7892167,"""female""","""Thái Nguyên""","""Standard""",2024-09-22,"""In-Store""",2024-09-22,373.425211,373.917091
7892168,"""female""","""Tây Ninh""","""Standard""",2024-09-22,"""iOS""",2024-09-22,373.425209,373.917091
8220123,"""female""","""Lâm Đồng""","""Standard""",2025-01-04,"""In-Store""",2025-01-04,269.074108,269.917091
8220128,"""female""","""Ninh Thuận""","""Standard""",2025-01-04,"""In-Store""",2025-01-21,269.073308,252.917091


In [6]:
df_transaction = read_parquet_transaction('/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/new-trans-data')
df_transaction.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,category_l1,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]",str,"decimal[38,4]",str,"decimal[38,4]"
"""4455000000001""",175750.0000,1,3875819,2024-04-13,"""In-Store""","""Tiền mặt""",390,9250.0000,"""Hóa mỹ phẩm cho bé""",185000.0000,"""Dầu sức khỏe""",0.0500
"""2950000000002""",53550.0000,12,4635706,2024-04-20,"""In-Store""","""Tiền mặt""",94,113400.0000,"""Sữa nước""",66000.0000,"""Sữa bột pha sẵn""",0.1886
"""0055000000001""",115000.0000,1,7411629,2024-04-21,"""In-Store""","""VietQR""",94,0.0000,"""Babycare""",115000.0000,"""Bình sữa, phụ kiện""",0.0000
"""2403000000001""",349000.0000,1,7296645,2024-04-13,"""In-Store""","""Tiền mặt""",612,0.0000,"""Sữa""",349000.0000,"""Enfa""",0.0000
"""0020010000440""",465000.0000,2,7411425,2024-04-21,"""SPE""","""Tiền mặt""",879,0.0000,"""Sữa""",465000.0000,"""Meiji""",0.0000


In [7]:
df_transaction.select([
    pl.col("created_date").min().alias("min_created_date"),
    pl.col("created_date").max().alias("max_created_date"),
])

min_created_date,max_created_date
date,date
2024-01-01,2025-01-30


In [13]:
df_item = read_parquet_item('./Pipeline-training2/preprocessed-dataset')
df_item.head()

item_id,price,category_l1,category_l2,category_l3,category,item_type,gender_target_final,description_final,brand_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Dr.Brown's""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Bộ quần áo""","""Bé Gái""","""Không xác định""","""Con Cưng""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Thương hiệu khác""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""Không xác định""","""Không xác định""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries""","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""Không xác định""","""Không xác định""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries""","""12-36M"""


# Training Stage 1

In [6]:
import numpy as np
import pandas as pd
import polars as pl

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GridSearchCV

from tqdm.auto import tqdm  # <- thêm dòng này


/datastore/uittogether/tools/miniconda3/envs/MABe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Chuẩn bị transaction, sort thời gian, tách train/valid (2024)

In [28]:
# df_transaction: Polars DataFrame gốc

df_trx = (
    df_transaction
    .select(["customer_id", "item_id", "created_date"])
    .drop_nulls(["customer_id", "item_id", "created_date"])
    .with_columns(
        pl.col("created_date")
        .cast(pl.Datetime)              # chuyển Date/String → Datetime
        .alias("created_datetime")
    )
    .sort("created_datetime")          # sắp xếp thời gian tăng dần
)

print("Tổng số dòng transaction:", df_trx.height)
print(df_trx.head())
print(df_trx.dtypes)

Tổng số dòng transaction: 35729825
shape: (5, 4)
┌─────────────┬───────────────┬──────────────┬─────────────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ created_datetime    │
│ ---         ┆ ---           ┆ ---          ┆ ---                 │
│ i32         ┆ str           ┆ date         ┆ datetime[μs]        │
╞═════════════╪═══════════════╪══════════════╪═════════════════════╡
│ 1028293     ┆ 4048000000008 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1158000000007 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1606000000010 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1627000000005 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 409015      ┆ 2803000000012 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
└─────────────┴───────────────┴──────────────┴─────────────────────┘
[Int32, String, Date, Datetime(time_unit='us', time_zone=None)]


In [29]:
# Chỉ lấy năm 2024
df_trx_2024 = df_trx.filter(
    pl.col("created_datetime").dt.year() == 2024
)

# Train = tháng 1 → 11/2024
df_train_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month().is_between(1, 11, closed="both")
)

# Valid = tháng 12/2024
df_valid_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month() == 12
)

print("Số dòng train (Polars):", df_train_pl.height)
print("Số dòng valid (Polars):", df_valid_pl.height)


Số dòng train (Polars): 32680632
Số dòng valid (Polars): 3049193


In [30]:
import pandas as pd

df_train = df_train_pl.to_pandas()
df_valid = df_valid_pl.to_pandas()

print("Số dòng train (pandas):", len(df_train))
print("Số dòng valid (pandas):", len(df_valid))
print(df_train.head())
print(df_valid.head())
print("df_train dtypes:\n", df_train.dtypes)


Số dòng train (pandas): 32680632
Số dòng valid (pandas): 3049193
   customer_id        item_id created_date created_datetime
0      1028293  4048000000008   2024-01-01       2024-01-01
1       512190  1158000000007   2024-01-01       2024-01-01
2       512190  1606000000010   2024-01-01       2024-01-01
3       512190  1627000000005   2024-01-01       2024-01-01
4       409015  2803000000012   2024-01-01       2024-01-01
   customer_id        item_id created_date created_datetime
0      6981489  6382000000005   2024-12-01       2024-12-01
1      7092360  5427000000006   2024-12-01       2024-12-01
2      6274269  0007090000357   2024-12-01       2024-12-01
3      3192555  3436000000013   2024-12-01       2024-12-01
4      4177385  3953000000092   2024-12-01       2024-12-01
df_train dtypes:
 customer_id                  int32
item_id                     object
created_date        datetime64[ms]
created_datetime    datetime64[us]
dtype: object


## Kiểm tra số lượng cold-start

In [31]:
import pickle, json
from pathlib import Path

gt_path = "groundtruth.pkl"

# 1) thử load pickle
try:
    with open(gt_path, "rb") as f:
        gt = pickle.load(f)
except Exception:
    # 2) nếu không phải pickle thì thử json
    with open(gt_path, "r", encoding="utf-8") as f:
        gt = json.load(f)

# gt thường có dạng:
# - dict: {customer_id: [item_id, item_id, ...], ...}
# hoặc list các record
# xử lý 2 case phổ biến:
if isinstance(gt, dict):
    U_gt = set(gt.keys())
elif isinstance(gt, list):
    # nếu list dict có key 'customer_id'
    U_gt = set(r["customer_id"] for r in gt if "customer_id" in r)
else:
    raise ValueError(f"Không hỗ trợ format groundtruth: {type(gt)}")

n_gt = len(U_gt)
print("n_gt =", n_gt)

n_gt = 391900


In [32]:
import polars as pl

# đảm bảo created_date là Date/Datetime
df_trx_2024 = (
    df_trx
    .with_columns(
        pl.col("created_date").cast(pl.Datetime, strict=False)
    )
    .filter(
        (pl.col("created_date") >= pl.datetime(2024, 1, 1)) &
        (pl.col("created_date") <  pl.datetime(2025, 1, 1))
    )
)

U_2024 = set(df_trx_2024.select("customer_id").unique().to_series().to_list())
n_2024 = len(U_2024)
print("n_2024 =", n_2024)

n_2024 = 2442306


In [33]:
cold_users = U_gt - U_2024
n_cold = len(cold_users)
cold_rate = n_cold / n_gt if n_gt > 0 else 0.0

print("n_cold =", n_cold)
print("cold_rate =", cold_rate)


n_cold = 58180
cold_rate = 0.14845623883643785


In [ ]:
["len_test" : 28, #
"len_val" : 28,
"len_hist" : 120, # số ngày trước đó dùng để train
"len_recent" : 28,
"min_trans_items" : 35,
"session_window" : 1,
"train_set" : [1],
"top_trans_items" : 1000,
"N_trend" : 90,
"N_cand" : 10,
"min_coo" : 600,
"N_neg" : 10,
"n_iter" : 1,
"Topk" : 10,
"filter_bought_items" : false,
"filter_fashion" : false]

## Định nghĩa class training cho stage 1

In [27]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm


class ItemItemCFStage1(BaseEstimator):
    """
    Stage 1: Item-Item CF + TF-IDF + Cosine kNN

    New additions:
      (A) time-decayed frequency: tw_cnt_ui = sum(exp(-lambda_td * days_since_each_event))
          -> can be used as base weight option: weight_type="time_decay"
      (B) history contribution weighting in recommend:
          score += sim(i, j) * (ui_strength(u,i) ** strength_power)
    """

    def __init__(
        self,
        df_train,
        df_valid,
        df_item,
        user_col="customer_id",
        item_col="item_id",

        # base user-item freq
        weight_type="log_count",  # "binary" | "count" | "log_count" | "rel_freq" | "time_decay"

        # UI recency / monetary
        use_ui_recency=True,
        ui_recency_lambda=0.01,
        use_ui_monetary=True,

        # (A) time-decayed frequency on raw interactions
        use_time_decay_count=True,
        time_decay_lambda=0.05,  # lambda_td in tw_cnt_ui

        # Category preference (L1)
        use_cat_l1_pref=True,
        alpha_l1_cnt=0.10,
        alpha_l1_spent=0.10,
        alpha_l1_rec=0.10,
        l1_recency_lambda=0.01,

        # Category preference (L2)
        use_cat_l2_pref=True,
        alpha_l2_cnt=0.15,
        alpha_l2_spent=0.15,
        alpha_l2_rec=0.15,
        l2_recency_lambda=0.01,

        # CF
        n_neighbors=100,
        k_eval=1000,
        use_tqdm=True,

        # (B) use user-item strength to weight history contributions
        use_ui_strength_in_scoring=True,
        strength_power=1.0,  # gamma
    ):
        self.df_train = df_train
        self.df_valid = df_valid
        self.df_item = df_item

        self.user_col = user_col
        self.item_col = item_col

        self.weight_type = weight_type

        self.use_ui_recency = use_ui_recency
        self.ui_recency_lambda = ui_recency_lambda
        self.use_ui_monetary = use_ui_monetary

        self.use_time_decay_count = use_time_decay_count
        self.time_decay_lambda = time_decay_lambda

        self.use_cat_l1_pref = use_cat_l1_pref
        self.alpha_l1_cnt = alpha_l1_cnt
        self.alpha_l1_spent = alpha_l1_spent
        self.alpha_l1_rec = alpha_l1_rec
        self.l1_recency_lambda = l1_recency_lambda

        self.use_cat_l2_pref = use_cat_l2_pref
        self.alpha_l2_cnt = alpha_l2_cnt
        self.alpha_l2_spent = alpha_l2_spent
        self.alpha_l2_rec = alpha_l2_rec
        self.l2_recency_lambda = l2_recency_lambda

        self.n_neighbors = n_neighbors
        self.k_eval = k_eval
        self.use_tqdm = use_tqdm

        self.use_ui_strength_in_scoring = use_ui_strength_in_scoring
        self.strength_power = strength_power

    def _build_user_item_matrix(self):
        df = self.df_train.copy()

        # ===== (0) check tối thiểu =====
        if "created_datetime" not in df.columns:
            raise ValueError("df_train must contain `created_datetime` (datetime) for recency features.")

        df["created_datetime"] = pd.to_datetime(df["created_datetime"], errors="coerce")
        df = df.dropna(subset=[self.user_col, self.item_col, "created_datetime"])

        # ===== (1) item metadata: map item -> cat_l1, cat_l2 =====
        item_meta_cols = [self.item_col]
        if "category_l1" in self.df_item.columns:
            item_meta_cols.append("category_l1")
        if "category_l2" in self.df_item.columns:
            item_meta_cols.append("category_l2")

        df_item_small = self.df_item[item_meta_cols].drop_duplicates(subset=[self.item_col]).copy()
        df = df.merge(df_item_small, on=self.item_col, how="left")

        if "category_l1" in df.columns:
            df["category_l1"] = df["category_l1"].fillna("__UNK_CAT_L1__")
        else:
            df["category_l1"] = "__NO_CAT_L1__"

        if "category_l2" in df.columns:
            df["category_l2"] = df["category_l2"].fillna("__UNK_CAT_L2__")
        else:
            df["category_l2"] = "__NO_CAT_L2__"

        # ===== (2) spent per row nếu có price/quantity =====
        has_price = "price" in df.columns
        has_qty = "quantity" in df.columns

        if has_price:
            df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0.0)
        if has_qty:
            df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(1.0)
        else:
            df["quantity"] = 1.0

        df["spent_row"] = (df["price"] * df["quantity"]) if has_price else 0.0

        cutoff_dt = df["created_datetime"].max()

        # ===== (A) time-decayed per-row contribution (for tw_cnt_ui) =====
        if self.use_time_decay_count:
            days_since_row = (cutoff_dt - df["created_datetime"]).dt.days
            days_since_row = days_since_row.fillna(9999).clip(lower=0).astype(float)
            df["row_decay"] = np.exp(-self.time_decay_lambda * days_since_row).astype(np.float32)
        else:
            df["row_decay"] = 1.0

        # ===== (3) Aggregate user-item =====
        ui = (
            df.groupby([self.user_col, self.item_col], as_index=False)
              .agg(
                  freq_cnt=(self.item_col, "size"),            # số giao dịch
                  tw_cnt_ui=("row_decay", "sum"),              # (A) time-decayed count
                  total_spent_ui=("spent_row", "sum"),
                  last_purchase_dt=("created_datetime", "max"),
              )
        )

        # ===== (4) base weight =====
        if self.weight_type == "binary":
            base = np.ones(ui.shape[0], dtype=np.float32)

        elif self.weight_type == "count":
            base = ui["freq_cnt"].astype(float).to_numpy().astype(np.float32)

        elif self.weight_type == "log_count":
            base = np.log1p(ui["freq_cnt"].astype(float).to_numpy()).astype(np.float32)

        elif self.weight_type == "rel_freq":
            base_raw = ui["freq_cnt"].astype(float)
            user_total = ui.groupby(self.user_col)["freq_cnt"].transform("sum").clip(lower=1)
            base = (base_raw / user_total).to_numpy().astype(np.float32)

        elif self.weight_type == "time_decay":
            # (A) dùng trực tiếp tw_cnt_ui làm base
            base = ui["tw_cnt_ui"].astype(float).to_numpy().astype(np.float32)

        else:
            raise ValueError(f"Unknown weight_type: {self.weight_type}")

        ui["base_weight"] = base

        # ===== (5) UI recency factor (last interaction) =====
        if self.use_ui_recency:
            days_since_ui = (cutoff_dt - ui["last_purchase_dt"]).dt.days
            days_since_ui = days_since_ui.fillna(9999).clip(lower=0).astype(float)
            ui["ui_recency_factor"] = np.exp(-self.ui_recency_lambda * days_since_ui).astype(np.float32)
        else:
            ui["ui_recency_factor"] = 1.0

        # ===== (6) UI monetary factor =====
        if self.use_ui_monetary and has_price:
            ui["ui_monetary_factor"] = np.log1p(ui["total_spent_ui"].astype(float)).astype(np.float32)
        else:
            ui["ui_monetary_factor"] = 1.0

        # ===== (7) Join lại category của item cho ui =====
        ui = ui.merge(df_item_small, on=self.item_col, how="left")
        if "category_l1" in ui.columns:
            ui["category_l1"] = ui["category_l1"].fillna("__UNK_CAT_L1__")
        else:
            ui["category_l1"] = "__NO_CAT_L1__"

        if "category_l2" in ui.columns:
            ui["category_l2"] = ui["category_l2"].fillna("__UNK_CAT_L2__")
        else:
            ui["category_l2"] = "__NO_CAT_L2__"

        # ===== (8) User totals for normalization =====
        user_tot = (
            df.groupby(self.user_col, as_index=False)
              .agg(
                  u_total_cnt=(self.item_col, "size"),
                  u_total_spent=("spent_row", "sum"),
              )
        )
        user_tot["u_total_cnt"] = user_tot["u_total_cnt"].clip(lower=1).astype(float)
        user_tot["u_total_spent"] = user_tot["u_total_spent"].astype(float)

        ui = ui.merge(user_tot, on=self.user_col, how="left")
        ui["u_total_cnt"] = ui["u_total_cnt"].fillna(1.0).astype(float)
        ui["u_total_spent"] = ui["u_total_spent"].fillna(0.0).astype(float)

        # ===== (9) L1 preference =====
        if self.use_cat_l1_pref:
            u_l1 = (
                df.groupby([self.user_col, "category_l1"], as_index=False)
                  .agg(
                      l1_cnt=(self.item_col, "size"),
                      l1_spent=("spent_row", "sum"),
                      l1_last_dt=("created_datetime", "max"),
                  )
            )
            ui = ui.merge(u_l1, on=[self.user_col, "category_l1"], how="left")
            ui["l1_cnt"] = ui["l1_cnt"].fillna(0.0).astype(float)
            ui["l1_spent"] = ui["l1_spent"].fillna(0.0).astype(float)

            l1_days = (cutoff_dt - ui["l1_last_dt"]).dt.days
            l1_days = l1_days.fillna(9999).clip(lower=0).astype(float)

            l1_cnt_norm = np.log1p(ui["l1_cnt"]) / np.log1p(ui["u_total_cnt"])
            l1_spent_norm = np.log1p(ui["l1_spent"]) / (np.log1p(ui["u_total_spent"]) + 1e-9)
            l1_rec_term = np.exp(-self.l1_recency_lambda * l1_days)

            ui["pref_l1"] = (
                self.alpha_l1_cnt * l1_cnt_norm
                + self.alpha_l1_spent * l1_spent_norm
                + self.alpha_l1_rec * l1_rec_term
            ).astype(np.float32)
        else:
            ui["pref_l1"] = 0.0

        # ===== (10) L2 preference =====
        if self.use_cat_l2_pref:
            u_l2 = (
                df.groupby([self.user_col, "category_l2"], as_index=False)
                  .agg(
                      l2_cnt=(self.item_col, "size"),
                      l2_spent=("spent_row", "sum"),
                      l2_last_dt=("created_datetime", "max"),
                  )
            )
            ui = ui.merge(u_l2, on=[self.user_col, "category_l2"], how="left")
            ui["l2_cnt"] = ui["l2_cnt"].fillna(0.0).astype(float)
            ui["l2_spent"] = ui["l2_spent"].fillna(0.0).astype(float)

            l2_days = (cutoff_dt - ui["l2_last_dt"]).dt.days
            l2_days = l2_days.fillna(9999).clip(lower=0).astype(float)

            l2_cnt_norm = np.log1p(ui["l2_cnt"]) / np.log1p(ui["u_total_cnt"])
            l2_spent_norm = np.log1p(ui["l2_spent"]) / (np.log1p(ui["u_total_spent"]) + 1e-9)
            l2_rec_term = np.exp(-self.l2_recency_lambda * l2_days)

            ui["pref_l2"] = (
                self.alpha_l2_cnt * l2_cnt_norm
                + self.alpha_l2_spent * l2_spent_norm
                + self.alpha_l2_rec * l2_rec_term
            ).astype(np.float32)
        else:
            ui["pref_l2"] = 0.0

        # ===== (11) Final w_ui =====
        pref_sum = (ui["pref_l1"].values.astype(np.float32) + ui["pref_l2"].values.astype(np.float32))

        w_ui = (
            ui["base_weight"].values.astype(np.float32)
            * ui["ui_recency_factor"].values.astype(np.float32)
            * ui["ui_monetary_factor"].values.astype(np.float32)
            * (1.0 + pref_sum)
        ).astype(np.float32)

        ui["value"] = w_ui

        # ===== (12) Encode user/item -> index & CSR =====
        user_cat = ui[self.user_col].astype("category")
        item_cat = ui[self.item_col].astype("category")

        self.user_index_to_id_ = list(user_cat.cat.categories)
        self.item_index_to_id_ = list(item_cat.cat.categories)

        self.user_id_to_index_ = {uid: idx for idx, uid in enumerate(self.user_index_to_id_)}
        self.item_id_to_index_ = {iid: idx for idx, iid in enumerate(self.item_index_to_id_)}

        user_codes = user_cat.cat.codes.to_numpy()
        item_codes = item_cat.cat.codes.to_numpy()
        data = ui["value"].to_numpy(dtype=np.float32)

        n_users = len(self.user_index_to_id_)
        n_items = len(self.item_index_to_id_)

        ui_matrix = csr_matrix(
            (data, (user_codes, item_codes)),
            shape=(n_users, n_items),
            dtype=np.float32
        )

        # store ui_matrix for (B) scoring weights
        self.ui_matrix_ = ui_matrix

        # history (set) for filtering + popularity fallback
        self.user_history_ = {}
        for u in range(n_users):
            start, end = ui_matrix.indptr[u], ui_matrix.indptr[u + 1]
            self.user_history_[u] = set(ui_matrix.indices[start:end])

        item_pop = np.asarray(ui_matrix.sum(axis=0)).ravel()
        self.popular_item_indices_ = np.argsort(-item_pop)

        return ui_matrix

    def _build_item_neighbors(self, ui_matrix):
        self.tfidf_ = TfidfTransformer(norm="l2", use_idf=True, sublinear_tf=True)
        ui_tfidf = self.tfidf_.fit_transform(ui_matrix)

        X_items = ui_tfidf.T

        self.nn_model_ = NearestNeighbors(
            n_neighbors=self.n_neighbors + 1,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1,
        )
        self.nn_model_.fit(X_items)

        distances, indices = self.nn_model_.kneighbors(X_items, return_distance=True)
        sims = 1.0 - distances

        self.item_neighbors_ = indices[:, 1:]
        self.item_neighbor_sims_ = sims[:, 1:]

    def _recommend_for_user_index(self, user_idx, top_k=None, allow_repeat=True, backfill_popular=True):
        if top_k is None:
            top_k = self.k_eval

        # cold-start
        if user_idx not in self.user_history_ or len(self.user_history_[user_idx]) == 0:
            return list(self.popular_item_indices_[:top_k])

        ui_mat = self.ui_matrix_
        start, end = ui_mat.indptr[user_idx], ui_mat.indptr[user_idx + 1]
        hist_items = ui_mat.indices[start:end]
        hist_weights = ui_mat.data[start:end]  # w_ui for each (u,i)

        history_set = self.user_history_.get(user_idx, set())

        candidate_scores = {}

        # (B) weight history contributions by ui_strength
        for item_i, w_ui in zip(hist_items, hist_weights):
            neighbors = self.item_neighbors_[item_i]
            sims = self.item_neighbor_sims_[item_i]

            if self.use_ui_strength_in_scoring:
                contrib = float(np.power(max(w_ui, 0.0), self.strength_power))
            else:
                contrib = 1.0

            for nbr_idx, sim in zip(neighbors, sims):
                if (not allow_repeat) and (nbr_idx in history_set):
                    continue
                candidate_scores[nbr_idx] = candidate_scores.get(nbr_idx, 0.0) + float(sim) * contrib

        if not candidate_scores:
            return list(self.popular_item_indices_[:top_k])

        sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
        rec = [idx for idx, _ in sorted_candidates]

        if len(rec) >= top_k:
            return rec[:top_k]

        if backfill_popular:
            rec_set = set(rec)
            for pop_i in self.popular_item_indices_:
                if pop_i not in rec_set:
                    rec.append(pop_i)
                    rec_set.add(pop_i)
                if len(rec) == top_k:
                    break

        return rec

    def fit(self, X=None, y=None):
        ui_matrix = self._build_user_item_matrix()
        self._build_item_neighbors(ui_matrix)
        return self

    def score(self, k=None, filter=True):
        if not hasattr(self, "user_history_"):
            raise RuntimeError("Model chưa fit. Hãy gọi fit() trước score().")

        if self.df_valid.empty:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        if k is None:
            k = self.k_eval

        df_val = self.df_valid[[self.user_col, self.item_col]].drop_duplicates().copy()

        item_cat_val = pd.Categorical(
            df_val[self.item_col],
            categories=self.item_index_to_id_
        )
        df_val["item_idx"] = item_cat_val.codes

        df_val = df_val[df_val["item_idx"] != -1]
        if df_val.empty:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        user_to_gt = (
            df_val.groupby(self.user_col)["item_idx"]
            .apply(lambda s: set(s.to_list()))
            .to_dict()
        )

        user_id_to_hist = {
            self.user_index_to_id_[u_idx]: items
            for u_idx, items in self.user_history_.items()
        }

        recalls, hits = [], []

        iterator = user_to_gt.items()
        if self.use_tqdm:
            iterator = tqdm(iterator, total=len(user_to_gt),
                            desc=f"Eval Stage1 @K={k} (filter={filter})",
                            leave=False)

        for user_id, gt_items in iterator:
            if user_id not in self.user_id_to_index_:
                continue

            hist_items = user_id_to_hist.get(user_id, set())
            relevant_items = (gt_items - hist_items) if filter else gt_items
            if len(relevant_items) == 0:
                continue

            user_idx = self.user_id_to_index_[user_id]

            # IMPORTANT: nếu filter=True (new-item recall) thì nên allow_repeat=False
            rec_items = self._recommend_for_user_index(
                user_idx,
                top_k=k,
                allow_repeat=(not filter)
            )

            if not rec_items:
                continue

            inter = set(rec_items) & relevant_items
            recalls.append(len(inter) / len(relevant_items))
            hits.append(1.0 if len(inter) > 0 else 0.0)

        if len(recalls) == 0:
            return {"recall": 0.0, "hit": 0.0, "n_users_eval": 0}

        return {
            "recall": float(np.mean(recalls)),
            "hit": float(np.mean(hits)),
            "n_users_eval": len(recalls),
        }

    def recommend_for_user_id(self, user_id, top_k=None, allow_repeat=True):
        if top_k is None:
            top_k = self.k_eval

        if user_id in self.user_id_to_index_:
            uidx = self.user_id_to_index_[user_id]
            rec_idx = self._recommend_for_user_index(uidx, top_k=top_k, allow_repeat=allow_repeat)
        else:
            rec_idx = list(self.popular_item_indices_[:top_k])

        return [self.item_index_to_id_[i] for i in rec_idx]

## Training-quick train

In [24]:
df_item_pd = df_item.to_pandas()


In [28]:
best_cfg = {
    "weight_type": "rel_freq",
    "n_neighbors": 100,

    "use_ui_recency": True,
    "ui_recency_lambda": 0.01,

    "use_cat_l1_pref": True,
    "alpha_l1_cnt": 0.05,
    "alpha_l1_spent": 0.05,
    "alpha_l1_rec": 0.1,

    "use_cat_l2_pref": False,   # đã tắt
    "alpha_l2_cnt": 0.15,
    "alpha_l2_spent": 0.1,
    "alpha_l2_rec": 0.1,
}
K_EVAL = 200  # đúng yêu cầu của bạn: 200 candidate

model = ItemItemCFStage1(
    df_train=df_train,
    df_valid=df_valid,
    df_item=df_item_pd,
    user_col="customer_id",
    item_col="item_id",
    k_eval=K_EVAL,
    **best_cfg
)

model.fit()



,df_train,cus...s x 4 columns]
,df_valid,cust...s x 4 columns]
,df_item,... x 11 columns]
,user_col,'customer_id'
,item_col,'item_id'
,weight_type,'rel_freq'
,use_ui_recency,True
,ui_recency_lambda,0.01
,use_ui_monetary,True
,use_time_decay_count,True
,time_decay_lambda,0.05


In [29]:
metric_full = model.score(k=1000, filter=False)
print(metric_full)

{'recall': 0.8103302491053823, 'hit': 0.9469740761257898, 'n_users_eval': 498228}


## Traing - random search

In [14]:
import random
from itertools import product
import json
import os
import joblib
from datetime import datetime


In [15]:
param_space = {
    "weight_type": ["log_count", "rel_freq"],

    # CF
    "n_neighbors": [50, 100, 200],

    # UI recency
    "use_ui_recency": [True],
    "ui_recency_lambda": [0.005, 0.01, 0.02],

    # Category L1
    "use_cat_l1_pref": [True],
    "alpha_l1_cnt": [0.05, 0.1, 0.2],
    "alpha_l1_spent": [0.05, 0.1],
    "alpha_l1_rec": [0.05, 0.1],

    # Category L2
    "use_cat_l2_pref": [False, True],
    "alpha_l2_cnt": [0.1, 0.15],
    "alpha_l2_spent": [0.1],
    "alpha_l2_rec": [0.1],
}


In [16]:
def sample_params(param_space, n_samples, tried_configs):
    keys = list(param_space.keys())
    samples = []

    while len(samples) < n_samples:
        cfg = {k: random.choice(param_space[k]) for k in keys}
        key = json.dumps(cfg, sort_keys=True)

        if key not in tried_configs:
            samples.append(cfg)
            tried_configs.add(key)

    return samples


In [17]:
SAVE_DIR = "stage1_random_search"
os.makedirs(SAVE_DIR, exist_ok=True)

tried_configs = set()
results = []

N_TRIALS = 20          # tuỳ GPU/CPU
K_EVAL = 200           # đúng yêu cầu của bạn

best_recall = -1
best_model = None
best_cfg = None


In [ ]:
df_item_pd = df_item.to_pandas()


In [ ]:
for trial, cfg in enumerate(sample_params(param_space, N_TRIALS, tried_configs), 1):

    print(f"\n=== Trial {trial}/{N_TRIALS} ===")
    print(cfg)

    model = ItemItemCFStage1(
        df_train=df_train,
        df_valid=df_valid,
        df_item=df_item_pd,
        user_col="customer_id",
        item_col="item_id",
        k_eval=K_EVAL,
        **cfg
    )

    model.fit()

    metrics = model.score(k=K_EVAL, filter=True)
    recall = metrics["recall"]
    hit = metrics["hit"]

    print(f"Recall@{K_EVAL}: {recall:.4f} | Hit@{K_EVAL}: {hit:.4f}")

    record = {
        "trial": trial,
        "recall": recall,
        "hit": hit,
        "params": cfg,
    }
    results.append(record)

    # ===== Save nếu tốt hơn best =====
    if recall > best_recall:
        best_recall = recall
        best_model = model
        best_cfg = cfg

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_path = f"{SAVE_DIR}/best_stage1_recall_{recall:.4f}_k{K_EVAL}_{ts}.pkl"

        joblib.dump(model, model_path)

        with open(f"{SAVE_DIR}/best_params.json", "w") as f:
            json.dump(best_cfg, f, indent=2)

        print(f"🔥 NEW BEST MODEL SAVED → {model_path}")


In [ ]:
import pandas as pd

df_results = pd.DataFrame(results).sort_values("recall", ascending=False)
df_results.to_csv(f"{SAVE_DIR}/random_search_results.csv", index=False)

print("Top 5 configs:")
display(df_results.head())


In [30]:
import joblib
# best_model: kết quả tốt nhất từ Stage 1 (ItemItemCFStage1)
joblib.dump(model, "stage1_item_item_cf.pkl")

['stage1_item_item_cf.pkl']

# Stage 2

In [31]:
import numpy as np

def precision_at_k(pred, gt, hist, filter_bought_items=True, K=10): # prediction, ground-truth, history items, candidate items
    precisions = []
    ideal_precs = []
    ncold_start = 0
    cold_start_users = []
    nusers = len(gt.keys())
    for user in gt.keys():
        if (user not in hist) or (user not in pred):
            ncold_start += 1
            cold_start_users.append(user) # THINKING: để giảm cold start có thể tăng khoảng HISTORY
            continue
        gt_items = gt[user]
        relevant_items = set(gt_items)
        if filter_bought_items:
            relevant_items -= set(hist[user])
        # Compute precision@k
        hits = len(set(pred[user][:K]) & relevant_items)
        precisions.append(hits / K)
    return np.mean(precisions), cold_start_users


## Import, load ground truth, chuẩn bị lịch sử (hist) cho Stage 2

In [32]:
import pickle
import json
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 1) Load ground truth tháng 01/2025
gt_raw = pd.read_pickle("groundtruth.pkl")

# Giả định gt_raw là dict: {user_id: [item1, item2, ...]}
if isinstance(gt_raw, dict):
    gt = gt_raw
elif isinstance(gt_raw, pd.Series):
    gt = gt_raw.to_dict()
else:
    # Nếu format khác, bạn in ra để chỉnh tay:
    print("groundtruth.pkl format:", type(gt_raw))
    # TODO: chỉnh parse cho đúng cấu trúc của bạn
    raise ValueError("Không biết format groundtruth.pkl, cần chỉnh lại parsing.")

# 2) Chuẩn bị lịch sử mua trước 2025-01-01 (hist) từ df_transaction (Polars)
#    đây là lịch sử để filter_bought_items trong precision_at_k

cutoff_test = pd.Timestamp("2025-01-01")

df_trx_hist = (
    df_transaction
    .select(["customer_id", "item_id", "created_date"])
    .drop_nulls(["customer_id", "item_id", "created_date"])
    .with_columns(
        pl.col("created_date").cast(pl.Datetime).alias("created_datetime")
    )
    .filter(pl.col("created_datetime") < cutoff_test)
)

df_trx_hist_pd = df_trx_hist.to_pandas()

hist = (
    df_trx_hist_pd
    .groupby("customer_id")["item_id"]
    .apply(lambda s: list(set(s.tolist())))
    .to_dict()
)

print("Số user trong ground truth:", len(gt))
print("Số user có lịch sử trước 2025-01-01:", len(hist))

Số user trong ground truth: 391900
Số user có lịch sử trước 2025-01-01: 2442306


## Sinh candidate từ Stage 1 và lưu xuống file

In [33]:
# best_model = joblib.load("stage1_item_item_cf.pkl")
best_model = model
K_cand = 1000  # số candidate Stage 1 per user cho Stage 2 (bạn có thể đổi 100/300/... tùy ý)

stage1_candidates = {}

for user_id in tqdm(gt.keys(), desc="Generate Stage1 candidates"):
    cand_items = best_model.recommend_for_user_id(user_id, top_k=K_cand)
    stage1_candidates[user_id] = cand_items

# Lưu lại để reuse
with open("stage1_candidates.pkl", "wb") as f:
    pickle.dump(stage1_candidates, f)

print("Số user có candidate:", len(stage1_candidates))


Generate Stage1 candidates: 100%|██████████| 391900/391900 [08:03<00:00, 810.89it/s] 


Số user có candidate: 391900


## Xây tập dữ liệu ranking cho LightGBM (Stage 2)

Chuẩn bị các feature đã engineering được

In [ ]:
brand_segment_df = pl.read_parquet("./new-feature/brand_segment.parquet")

Chuẩn bị user/item feature

In [34]:
# Chuyển df_user, df_item sang pandas
df_user_pd = df_user.to_pandas()
df_item_pd = df_item.to_pandas()

# Rút gọn các cột cần dùng (có thể thêm/bớt tùy bạn)
user_feat_cols = [
    "customer_id",
    "gender",
    "province",
    "membership",
    "user_age_days",
    "days_since_install",
    "days_since_last_sync",
]

df_user_feat = df_user_pd[user_feat_cols].drop_duplicates(subset=["customer_id"])

item_feat_cols = [
    "item_id",
    "price",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
    # có thể thêm: "item_type", "gender_target_final"
]

df_item_feat = df_item_pd[item_feat_cols].drop_duplicates(subset=["item_id"])


Build DataFrame ranking

In [ ]:
rows = []

for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Build ranking rows"):
    gt_items = set(gt.get(user_id, []))

    for rank_pos, item_id in enumerate(cand_items):
        label = 1 if item_id in gt_items else 0
        rows.append(
            {
                "customer_id": user_id,
                "item_id": item_id,
                "label": label,
                "stage1_rank": rank_pos,  # vị trí trong output Stage1
            }
        )

df_rank = pd.DataFrame(rows)
print("Số dòng ranking:", len(df_rank))
print(df_rank.head())


Build ranking rows: 100%|██████████| 391900/391900 [02:42<00:00, 2418.93it/s]


Join user/item features vào df_rank

In [ ]:
df_rank = df_rank.merge(df_user_feat, on="customer_id", how="left")
df_rank = df_rank.merge(df_item_feat, on="item_id", how="left")

# Một số fillna cơ bản
df_rank["price"] = df_rank["price"].fillna(0.0)
df_rank["stage1_rank"] = df_rank["stage1_rank"].fillna(K_cand).astype(int)

# With categorical columns as category dtype
cat_cols = [
    "gender",
    "province",
    "membership",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
]

for c in cat_cols:
    if c in df_rank.columns:
        df_rank[c] = df_rank[c].astype("category")


## Tách train/valid cho LightGBM, train bằng GPU và lưu model

In [ ]:
# Lấy danh sách user có label trong ranking
users_all = df_rank["customer_id"].unique()

train_users, valid_users = train_test_split(
    users_all, test_size=0.2, random_state=42
)

# df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
# df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

# print("Train users:", len(train_users), "Valid users:", len(valid_users))
# print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


In [ ]:
# 1) Xác định lại list feature
feature_cols = [
    c for c in df_rank.columns
    if c not in ["label", "customer_id", "item_id"]
]

# 2) Khai báo các cột categorical theo tên (nếu tồn tại trong df_rank)
cat_cols = [
    "gender",
    "province",
    "membership",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
]

cat_feature_names = [c for c in cat_cols if c in feature_cols]

# 3) Xử lý cột categorical: fill NA bằng '__MISSING__' rồi cast sang category
for c in cat_feature_names:
    # chuyển sang string, fill missing, sau đó cast về category
    df_rank[c] = df_rank[c].astype("string").fillna("__MISSING__").astype("category")

# 4) Các cột numeric: ép về số, fillna(0.0)
numeric_cols = [col for col in feature_cols if col not in cat_feature_names]

for col in numeric_cols:
    df_rank[col] = pd.to_numeric(df_rank[col], errors="coerce")
    df_rank[col] = df_rank[col].fillna(0.0)

# Không dùng df_rank.fillna(0.0) toàn bảng nữa!
print(df_rank[feature_cols].dtypes)

# 6) Tách lại train/valid như trước (nếu bạn đã tách rồi, chỉ cần update df_train_rank, df_valid_rank)
df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

X_train = df_train_rank[feature_cols]
y_train = df_train_rank["label"]

X_valid = df_valid_rank[feature_cols]
y_valid = df_valid_rank["label"]

# 7) Tạo danh sách index cho categorical_feature
cat_feature_indices = [feature_cols.index(c) for c in cat_feature_names]

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=cat_feature_indices,
    free_raw_data=False,
)

valid_data = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=cat_feature_indices,
    free_raw_data=False,
)

print("Train users:", len(train_users), "Valid users:", len(valid_users))
print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


stage1_rank                int64
gender                  category
province                category
membership              category
user_age_days            float64
days_since_install       float64
days_since_last_sync     float64
price                    float64
category_l1             category
category_l2             category
brand                   category
age_group_final         category
dtype: object
Train users: 313520 Valid users: 78380
Train rows: 222202093 Valid rows: 55682338


In [ ]:
params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "verbosity": -1,
    # GPU
    "device": "gpu",      # bản mới dùng "device_type"
    "gpu_device_id": 7,      # nếu cần chỉ định GPU ID
}

evals_result = {}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.record_evaluation(evals_result),
    lgb.log_evaluation(period=50),
]

bst = lgb.train(
    params,
    train_data,
    num_boost_round=1,
    valid_sets=[train_data, valid_data],
    valid_names=["train", "valid"],
    callbacks=callbacks,
)

bst.save_model("lgb_stage2_ranking.txt", num_iteration=bst.best_iteration)

print("Best iteration:", bst.best_iteration)

Training until validation scores don't improve for 50 rounds


In [ ]:
K_eval_final = 10  # K cho precision@K

pred = {}

for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Predict Stage2 scores"):
    if len(cand_items) == 0:
        continue

    # Subset candidate rows cho user này từ df_rank
    # (để đảm bảo feature engineering giống lúc train)
    mask = (df_rank["customer_id"] == user_id) & (df_rank["item_id"].isin(cand_items))
    df_user_cand = df_rank.loc[mask, ["customer_id", "item_id"] + feature_cols].copy()

    if df_user_cand.empty:
        continue

    X_user = df_user_cand[feature_cols]
    scores = bst.predict(X_user, num_iteration=bst.best_iteration)

    df_user_cand["score"] = scores

    # Sort theo score giảm dần, lấy top K_eval_final
    df_user_cand_sorted = df_user_cand.sort_values("score", ascending=False)
    top_items = df_user_cand_sorted["item_id"].tolist()[:K_eval_final]

    pred[user_id] = top_items

# Tính precision@K theo hàm bạn cung cấp
prec, cold_users = precision_at_k(
    pred=pred,
    gt=gt,
    hist=hist,
    filter_bought_items=True,
    K=K_eval_final,
)

print(f"Precision@{K_eval_final}: {prec:.4f}")
print("Số user bị xem là cold-start theo hàm precision:", len(cold_users))
